# Training Techniques & Hyperparameter Search

Companion notebook for the [Training Techniques lesson](https://ml-viz-ruby.vercel.app/courses/model-evaluation/03-training-techniques).

**The idea in one sentence.** How you *run* training — when to stop, how to schedule
the learning rate, and how to search hyperparameters — often matters as much as the
model itself.

What this notebook builds and validates:

- **Early stopping** — halt when validation loss stops improving, keeping the best
  checkpoint (the cheapest regularizer there is).
- **Learning-rate schedules** — warmup + decay beats a constant rate.
- **Grid vs random search** — in high dimensions, random search covers the
  important axes far better than grid.

We **validate early stopping and the surprising efficiency of random search**, then
cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = '#94a3b8'
plt.rcParams['xtick.color']      = '#94a3b8'
plt.rcParams['ytick.color']      = '#94a3b8'
plt.rcParams['axes.edgecolor']   = '#2e3347'
plt.rcParams['grid.color']       = '#2e3347'

BRAND  = '#818cf8'
TEAL   = '#14b8a6'
YELLOW = '#f59e0b'
ROSE   = '#f43f5e'

rng = np.random.default_rng(42)

## 1. Early Stopping

Training loss decreases monotonically, but validation loss starts rising after ~40 epochs — this is when to stop.

In [ ]:
epochs = np.arange(1, 101)

# Simulated loss curves
train_loss = 1.0 * np.exp(-epochs / 30) + 0.05 + rng.normal(0, 0.005, 100)
val_noise  = rng.normal(0, 0.01, 100)
val_loss   = (1.0 * np.exp(-epochs / 30) + 0.10 +
              0.001 * np.maximum(0, epochs - 40) ** 1.3 + val_noise)

best_epoch = np.argmin(val_loss) + 1

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs, train_loss, color=BRAND,  linewidth=2, label='Train loss')
ax.plot(epochs, val_loss,   color=YELLOW, linewidth=2, label='Validation loss')
ax.axvline(best_epoch, color=TEAL, linestyle='--', linewidth=1.5,
           label=f'Best epoch = {best_epoch} (early stop here)')
ax.fill_between(epochs[best_epoch-1:], train_loss[best_epoch-1:], val_loss[best_epoch-1:],
                alpha=0.12, color=ROSE)
ax.annotate('Overfitting\ngap grows', xy=(75, val_loss[74]),
            xytext=(82, val_loss[74] + 0.05), color=ROSE,
            arrowprops=dict(arrowstyle='->', color=ROSE))
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Early stopping: restore weights from best validation epoch', color='white')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best epoch: {best_epoch}  val_loss={val_loss[best_epoch-1]:.4f}')
print(f'Final epoch: {len(epochs)}  val_loss={val_loss[-1]:.4f}')
print(f'Generalization gap at final epoch: {val_loss[-1] - train_loss[-1]:.4f}')

### Validate: early stopping recovers the best checkpoint

Early stopping watches validation loss and stops after `patience` epochs without
improvement, restoring the epoch with the lowest val loss. On a classic U-shaped
val curve, it must land at (or just after) the true minimum — not the last, overfit
epoch.

In [ ]:
import numpy as np
val = np.array([1.0, 0.8, 0.6, 0.5, 0.52, 0.55, 0.6, 0.7])   # min at index 3
def early_stop(val_losses, patience=3):
    best, best_i, wait = np.inf, 0, 0
    for i, v in enumerate(val_losses):
        if v < best:
            best, best_i, wait = v, i, 0
        else:
            wait += 1
            if wait >= patience:
                return best_i
    return best_i
stop_at = early_stop(val, patience=3)
print(f'validation losses: {val.tolist()}')
print(f'early stopping restored epoch {stop_at} (true minimum at {int(np.argmin(val))})')
assert stop_at == int(np.argmin(val)), 'early stopping must return the min-val epoch'
print('✅ early stopping keeps the best checkpoint, not the last (overfit) one')

## 2. Learning Rate Schedules

Four common schedules over 100 epochs: step decay, cosine annealing, reduce-on-plateau, and warm-up + cosine.

In [ ]:
T = 100
ts = np.arange(T)
lr_max, lr_min = 0.1, 1e-4

# Step decay: reduce by 0.1 every 30 epochs
step_decay = lr_max * (0.1 ** (ts // 30))

# Cosine annealing
cosine = lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * ts / T))

# Warm-up (10 epochs) then cosine
warmup = 10
warmup_cosine = np.where(
    ts < warmup,
    lr_max * ts / warmup,
    lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * (ts - warmup) / (T - warmup)))
)

# Reduce on plateau (simulated: reduce by 0.5 at epochs 40, 70)
rop = np.ones(T) * lr_max
for e in [40, 70]:
    rop[e:] *= 0.5

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ts, step_decay,    color=BRAND,  linewidth=2, label='Step decay (×0.1 every 30)')
ax.plot(ts, cosine,        color=TEAL,   linewidth=2, label='Cosine annealing')
ax.plot(ts, warmup_cosine, color=YELLOW, linewidth=2, label='Warm-up (10) + cosine')
ax.plot(ts, rop,           color=ROSE,   linewidth=2, linestyle='--', label='Reduce on plateau')
ax.axvline(warmup, color=YELLOW, linestyle=':', alpha=0.5)
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning rate (log scale)')
ax.set_title('Learning Rate Schedules', color='white')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**What to notice — schedules beat a constant LR.** A warmup avoids blowing up
early when gradients are large; the subsequent decay (cosine/step) lets the model
settle into a sharper minimum instead of bouncing around it. The area under a good
schedule is "big steps early, small steps late.\"

## 3. Grid Search vs Random Search

For a 2D problem where only the learning rate (x-axis) matters, random search explores far more distinct LR values than a grid.

In [ ]:
# Simulate 25 evaluations for each strategy
n_trials = 25
grid_side = 5  # 5x5 = 25 grid points

# Grid search: 5 LR × 5 batch size values
lr_grid = np.logspace(-5, -1, grid_side)
bs_grid = np.array([16, 64, 128, 256, 512])
grid_lr, grid_bs = np.meshgrid(lr_grid, bs_grid)
grid_lr = grid_lr.ravel()
grid_bs = grid_bs.ravel()

# Random search: 25 independent samples
rand_lr = np.exp(rng.uniform(np.log(1e-5), np.log(1e-1), n_trials))
rand_bs = rng.choice([16, 32, 64, 128, 256, 512], n_trials)

# "True" performance: only LR matters (Gaussian around best LR=1e-3)
def perf(lr): return np.exp(-((np.log10(lr) - np.log10(1e-3))**2) / 0.5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, lrs, bss, title in [
    (ax1, grid_lr, grid_bs, f'Grid search ({grid_side}×{grid_side}=25 pts)\n{grid_side} unique LR values'),
    (ax2, rand_lr, rand_bs, f'Random search (25 pts)\n{n_trials} unique LR values'),
]:
    perfs = perf(lrs)
    sc = ax.scatter(lrs, bss, c=perfs, cmap='plasma', s=80, vmin=0, vmax=1)
    ax.set_xscale('log')
    ax.set_xlabel('Learning rate')
    ax.set_ylabel('Batch size')
    ax.set_title(title, color='white')
    ax.axvline(1e-3, color=TEAL, linestyle='--', alpha=0.5, label='Optimal LR=1e-3')
    plt.colorbar(sc, ax=ax, label='Performance')
    ax.legend()

plt.suptitle('Grid vs Random Search — only LR matters, batch size is unimportant', color='white')
plt.tight_layout()
plt.show()

print(f'Grid search: {grid_side} unique LR values')
print(f'Random search: {n_trials} unique LR values')
print(f'Best grid performance:   {max(perf(grid_lr)):.4f}')
print(f'Best random performance: {max(perf(rand_lr)):.4f}')

### Validate: random search beats grid in high dimensions

If only a few hyperparameters actually matter, grid search wastes its budget
sampling the irrelevant ones at fixed values, while random search tries a *distinct*
value of every important parameter on every trial. We compare unique values explored
along one important axis for the same budget.

In [ ]:
rng_s = np.random.default_rng(0)
budget = 25
# grid: 5x5 over two params -> only 5 distinct values per axis
grid_vals = np.linspace(0, 1, 5)
grid_unique_axis1 = len(set(np.repeat(grid_vals, 5)))
# random: 25 draws -> up to 25 distinct values per axis
rand_axis1 = rng_s.uniform(0, 1, budget)
rand_unique_axis1 = len(np.unique(np.round(rand_axis1, 6)))
print(f'grid   ({budget} evals): {grid_unique_axis1} distinct values on the important axis')
print(f'random ({budget} evals): {rand_unique_axis1} distinct values on the important axis')
assert rand_unique_axis1 > grid_unique_axis1, 'random search explores more distinct values per axis'
print('\n✅ for the same budget random search resolves each important axis far more finely')

## 4. Bayesian Optimization (Illustrated)

A Gaussian Process surrogate is updated after each evaluation. The acquisition function (Expected Improvement) guides where to sample next — balancing exploration and exploitation.

In [ ]:
# Simulate a 1D Bayesian optimization scenario
# True objective (unknown to optimizer): bimodal with best at x≈0.7
def true_objective(x):
    return -(0.5 * np.exp(-((x - 0.7)**2) / 0.02) +
             0.3 * np.exp(-((x - 0.3)**2) / 0.01)) + 0.1 * rng.standard_normal()

# Initial observations
x_obs = np.array([0.1, 0.4, 0.9])
y_obs = np.array([true_objective(x) for x in x_obs])

# Simple GP posterior (closed-form for RBF kernel, noise=0.01)
def gp_posterior(x_new, x_obs, y_obs, length_scale=0.15, noise=0.05):
    def rbf(a, b): return np.exp(-((a[:, None] - b[None, :])**2) / (2 * length_scale**2))
    K    = rbf(x_obs, x_obs) + noise * np.eye(len(x_obs))
    k_s  = rbf(x_obs, x_new)
    k_ss = rbf(x_new, x_new)
    K_inv  = np.linalg.inv(K)
    mu     = k_s.T @ K_inv @ y_obs
    sigma2 = np.diag(k_ss - k_s.T @ K_inv @ k_s)
    return mu, np.sqrt(np.maximum(sigma2, 0))

x_grid = np.linspace(0, 1, 300)
mu, sigma = gp_posterior(x_grid, x_obs, y_obs)

# Expected Improvement acquisition function
from scipy.stats import norm
f_best = y_obs.min()
z  = (f_best - mu) / (sigma + 1e-9)
ei = sigma * (z * norm.cdf(z) + norm.pdf(z))
next_x = x_grid[np.argmax(ei)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# True objective
x_true = np.linspace(0, 1, 300)
y_true_obj = -(0.5 * np.exp(-((x_true - 0.7)**2) / 0.02) +
               0.3 * np.exp(-((x_true - 0.3)**2) / 0.01))
ax1.plot(x_true, y_true_obj, color='gray', linewidth=1, alpha=0.4, label='True objective')
ax1.plot(x_grid, mu, color=BRAND, linewidth=2, label='GP mean')
ax1.fill_between(x_grid, mu - 2*sigma, mu + 2*sigma, alpha=0.15, color=BRAND, label='GP ±2σ')
ax1.scatter(x_obs, y_obs, color=TEAL, s=80, zorder=6, label='Observations')
ax1.axvline(next_x, color=ROSE, linestyle='--', linewidth=1.5, label=f'Next query x={next_x:.2f}')
ax1.set_ylabel('Objective')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_title('GP surrogate model after 3 evaluations', color='white')

ax2.plot(x_grid, ei, color=YELLOW, linewidth=2)
ax2.axvline(next_x, color=ROSE, linestyle='--', linewidth=1.5)
ax2.fill_between(x_grid, 0, ei, alpha=0.2, color=YELLOW)
ax2.set_xlabel('Hyperparameter x')
ax2.set_ylabel('Expected Improvement')
ax2.set_title(f'Acquisition function — next query at x={next_x:.2f}', color='white')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## ✏️ Your turn

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **patience too small** | stops on a noisy val bump before the real minimum |
| **no LR schedule** | constant rate either diverges early or crawls late |
| **grid in high-D** | $m^d$ evaluations — intractable past 2–3 hyperparameters (demo) |
| **tuning on the test set** | HPO against the test split leaks; use a separate validation set |
| **not restoring best weights** | early stopping without checkpointing keeps the *worse* final epoch |

Demo: grid search's evaluation count explodes exponentially with dimension.

In [ ]:
# Grid search's cost EXPLODES with dimensions (the curse of dimensionality): m values
# per hyperparameter over d hyperparameters is m**d evaluations. Random search decouples
# budget from dimension entirely.
for d in [1, 2, 4, 8]:
    m = 5
    print(f'{d} hyperparameters, {m} values each: grid needs {m**d:>6} evals; '
          f'random search uses whatever budget you set')
print('\nGrid is only tractable for 2-3 hyperparameters; beyond that, random or Bayesian search wins.')

### Exercise 1 — `early_stopping_epoch(val_losses, patience=5)`

Implement early stopping logic:
- Track the best (lowest) validation loss seen so far
- Count how many consecutive epochs have passed without improvement
- When the count reaches `patience`, stop
- Return the **epoch index** (0-based) of the best validation loss

In [ ]:
def early_stopping_epoch(val_losses, patience=5):
    """
    Find the best epoch using early stopping logic.
    Returns the 0-based index of the best validation loss.
    """
    # TODO(you): iterate over val_losses, track best_loss and best_epoch,
    # increment patience_count when no improvement, stop when patience_count >= patience
    best_loss = float('inf')
    best_epoch = 0
    patience_count = 0
    for i, loss in enumerate(val_losses):
        ...  # fill in
    return best_epoch

In [ ]:
# Test 1: best at index 2, stops after 3 non-improving epochs
losses1 = [0.9, 0.7, 0.5, 0.6, 0.7, 0.8]
result1 = early_stopping_epoch(losses1, patience=3)
assert result1 == 2, f"Expected best_epoch=2, got {result1}"

# Test 2: monotonically decreasing — best is the last epoch
losses2 = [0.9, 0.7, 0.5, 0.3, 0.1]
result2 = early_stopping_epoch(losses2, patience=3)
assert result2 == 4, f"Expected best_epoch=4, got {result2}"

# Test 3: monotonically increasing with patience=2 — best is epoch 0
losses3 = [0.5, 0.6, 0.7, 0.8, 0.9]
result3 = early_stopping_epoch(losses3, patience=2)
assert result3 == 0, f"Expected best_epoch=0, got {result3}"

# Edge case: patience=0 — stop as soon as any epoch fails to improve
losses4 = [0.5, 0.6, 0.4, 0.3]
result4 = early_stopping_epoch(losses4, patience=0)
assert result4 == 0, f"Expected best_epoch=0 (patience=0 stops at the first non-improving epoch), got {result4}"

print(f"Test 1 best_epoch: {result1}  (expected 2)")
print(f"Test 2 best_epoch: {result2}  (expected 4)")
print(f"Test 3 best_epoch: {result3}  (expected 0)")
print(f"Test 4 (patience=0) best_epoch: {result4}  (expected 0)")
print("\u2705 Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def early_stopping_epoch(val_losses, patience=5):
    best_loss = float('inf')
    best_epoch = 0
    patience_count = 0
    for i, loss in enumerate(val_losses):
        if loss < best_loss:
            best_loss = loss
            best_epoch = i
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break
    return best_epoch
```

</details>

### Exercise 2 — `random_vs_grid_coverage(n_trials, ...)`

Compare how many **unique learning rate values** grid search vs random search explore.

- **Grid search:** form a `ceil(sqrt(n_trials)) × ceil(sqrt(n_trials))` grid; count distinct LR values
- **Random search:** sample `n_trials` random (lr, bs) pairs from log-uniform LR and categorical BS; count distinct LR values

Return `(n_unique_grid_lr, n_unique_random_lr)`.

In [ ]:
def random_vs_grid_coverage(n_trials, lr_range=(1e-5, 1e-1), bs_range=(16, 512), seed=42):
    """
    Returns (n_unique_grid_lr, n_unique_random_lr).
    """
    rng_c = np.random.default_rng(seed)
    import math

    # TODO(you): build the grid and random samples
    # Grid: side = ceil(sqrt(n_trials)); grid of lr_values × bs_values
    side = math.ceil(math.sqrt(n_trials))
    # lr_values = np.logspace(log10(lr_range[0]), log10(lr_range[1]), side)
    # random: sample n_trials lrs from log-uniform distribution

    n_unique_grid_lr   = ...
    n_unique_random_lr = ...
    return n_unique_grid_lr, n_unique_random_lr

In [ ]:
g, r = random_vs_grid_coverage(n_trials=25)
assert g is not None and r is not None, "returned None"
assert r > g, \
    f"Random search should cover more unique LR values than grid. Got grid={g}, random={r}"
print(f"Grid search unique LR values:   {g}  (should be sqrt(25)=5)")
print(f"Random search unique LR values: {r}  (should be 25)")
print("\u2705 Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def random_vs_grid_coverage(n_trials, lr_range=(1e-5, 1e-1), bs_range=(16, 512), seed=42):
    rng_c = np.random.default_rng(seed)
    import math
    side = math.ceil(math.sqrt(n_trials))

    # Grid: side distinct LR values
    lr_values = np.logspace(np.log10(lr_range[0]), np.log10(lr_range[1]), side)
    n_unique_grid_lr = len(lr_values)  # = side

    # Random: n_trials independent LR samples from log-uniform
    rand_lrs = np.exp(rng_c.uniform(np.log(lr_range[0]), np.log(lr_range[1]), n_trials))
    n_unique_random_lr = len(rand_lrs)  # all unique (continuous distribution)

    return n_unique_grid_lr, n_unique_random_lr
```

</details>

### Exercise 3 — `early_stopping(val_losses, patience, min_delta)` (DML 135)

Open-Deep-ML `135_implement-early-stopping-based-on-validation-loss` adds a
**`min_delta`** threshold on top of Exercise 1's patience counter: an epoch
only counts as an improvement if the loss drops by *more* than `min_delta`,
not just any decrease. It also returns **both** the stop epoch and the best
epoch, as a `(stop_epoch, best_epoch)` tuple.

- Track `best_loss` / `best_epoch`, starting from epoch 0.
- From epoch 1 onward: if `val_losses[epoch] < best_loss - min_delta`, that's
  an improvement — update `best_loss` / `best_epoch` and reset the
  non-improvement counter (`wait`).
- Otherwise increment `wait`; once `wait >= patience`, return
  `(epoch, best_epoch)` immediately.
- If training never triggers early stopping, return
  `(len(val_losses) - 1, best_epoch)`.

In [ ]:
from typing import Tuple

def early_stopping(val_losses: list, patience: int, min_delta: float) -> Tuple[int, int]:
    """
    DML 135: returns (stop_epoch, best_epoch).
    An epoch only counts as an improvement if it beats best_loss by more than min_delta.
    """
    best_loss = val_losses[0]
    best_epoch = 0
    wait = 0
    # TODO(you): loop from epoch 1, compare against best_loss - min_delta,
    # update best_loss/best_epoch/wait, and return (epoch, best_epoch) as soon
    # as wait >= patience. If the loop finishes without early stopping, return
    # (len(val_losses) - 1, best_epoch).
    return ...

In [ ]:
# DML's own tests
assert early_stopping([0.9, 0.8, 0.75, 0.77, 0.76, 0.77, 0.78], 2, 0.01) == (4, 2)
assert early_stopping([0.9, 0.8, 0.7, 0.6, 0.5], 2, 0.01) == (4, 4)
assert early_stopping([0.9, 0.8, 0.79, 0.78, 0.77], 2, 0.1) == (4, 2)
assert early_stopping([0.5, 0.4], 3, 0.01) == (1, 1)
assert early_stopping([0.5, 0.4, 0.4, 0.4, 0.4], 2, 0.01) == (3, 1)

# Edge case: patience=0 — stop at the very first non-improving epoch
assert early_stopping([0.5, 0.6, 0.4, 0.3], 0, 0.01) == (1, 0), \
    "patience=0 should stop at epoch 1 (the first non-improving epoch)"

print("\u2705 Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
from typing import Tuple

def early_stopping(val_losses, patience, min_delta):
    best_loss = val_losses[0]
    best_epoch = 0
    wait = 0
    for epoch in range(1, len(val_losses)):
        if val_losses[epoch] < best_loss - min_delta:
            best_loss = val_losses[epoch]
            best_epoch = epoch
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                return epoch, best_epoch
    return len(val_losses) - 1, best_epoch
```

</details>

## Key takeaways

- **Early stopping is free regularization:** stop when val loss plateaus, keep the
  best checkpoint (we verified it lands at the minimum).
- **Schedule the learning rate:** warmup + decay beats a constant rate — big steps
  early, small steps late.
- **Random > grid in high dimensions:** for the same budget, random search resolves
  each important axis far more finely (verified), and its cost is independent of
  dimension (unlike grid's $m^d$ blow-up).
- **Bayesian optimization** goes further by modelling the response surface — see the
  Bayesian Methods course.